# 4 · Read the actual results

Run after the evaluation and report stages. Published example results are also available in `reports/reference/`. The purpose is to understand the evidence, including cases where the simple baseline works better.

In [ ]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
from traffic_anomaly.paths import ROOT, RAW, RESULTS, FIGURES, config
from traffic_anomaly.data import load_split
cfg = config()
print("Project configuration found:", (ROOT / "configs/study.json").is_file())

## What was executed

In [ ]:
run = json.loads((RESULTS / "run_manifest.json").read_text())
display(pd.Series({k:run[k] for k in ["completed_at_utc","python","n_test_scenarios","n_metric_rows","n_seed_sensitivity_runs"]}).to_frame("value"))
display(pd.Series(run["packages"]).to_frame("version"))

## Compare scenario families

These are median, minimum and maximum recall across five injection seeds at the 1% validation alert target. Ranges describe seed variation; they are not confidence intervals for all future traffic.

In [ ]:
summary = pd.read_csv(RESULTS / "summary.csv")
display(summary[summary.severity == "severe"].round(3))
display(Image(filename=str(FIGURES / "02_recall_by_condition.png")))

## Check the alert burden too

If one method detects more changed hours but flags many more unchanged hours, recall alone does not settle which is preferable. The final test background rate can differ from the validation target.

In [ ]:
background = pd.read_csv(RESULTS / "original_alert_rates.csv")
display(background[background.calibration_target == cfg["primary_alert_rate"]].round(4))
display(Image(filename=str(FIGURES / "05_monthly_background_alerts.png")))

## Separate two kinds of randomness

Injection-seed results change where anomalies are placed. This next table instead changes the forest seed while the injected records stay fixed.

In [ ]:
seed_results = pd.read_csv(RESULTS / "model_seed_sensitivity.csv")
display(seed_results.groupby("family")[["recall","average_precision","background_alert_rate"]].agg(["min","median","max"]).round(3))

## Follow a result back to its input

`metrics.csv` records one method and threshold per row. Matching fingerprints confirm that two methods were evaluated on exactly the same altered sequence. `injected_changes.csv.gz` records the changed counts, while `scenario_manifest.csv` records the actual injection rate.

In [ ]:
metrics_table = pd.read_csv(RESULTS / "metrics.csv")
scenario = metrics_table.scenario.iloc[0]
display(metrics_table[(metrics_table.scenario == scenario) & (metrics_table.calibration_target == .01)][["scenario","method","fingerprint","n_injected","recall","precision","background_alert_rate"]])
changes = pd.read_csv(RESULTS / "injected_changes.csv.gz")
display(changes[changes.scenario == scenario].head())

## Explain it in your own words

1. Which method finds severe drops better in this experiment?
2. Does that method also generate more background alerts?
3. Why might flatlines be difficult without historical features?
4. What can we conclude about synthetic changes, and what remains unknown about real faults?

Read `reports/RESULTS.md` and [the methods guide](../docs/methods.md) for the interpretation. Changing settings after seeing these results creates an exploratory follow-up, not another untouched final test.